In [1]:
import langchain_openai

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [3]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain.retrievers import BM25Retriever

In [4]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o-mini")


In [5]:
def load_pdfs_from_folder(folder_path):
    """
    Belirtilen klasör altındaki tüm PDF dosyalarını yükler
    ve her bir sayfayı 'Document' nesnesi olarak döndürür.
    """
    all_docs = []
    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(".pdf"):
            full_path = os.path.join(folder_path, file_name)
            loader = PyPDFLoader(full_path)
            pdf_docs = loader.load()
            all_docs.extend(pdf_docs)
    return all_docs

In [6]:
def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    """
    Doküman listesini belirtilen boyutta parçalara ayırarak geri döndürür.
    """
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    
    splitted_docs = []
    for doc in documents:
        for chunk in text_splitter.split_text(doc.page_content):
            # Metadataları koruyabilir veya ek alanlar ekleyebilirsiniz
            splitted_docs.append(doc.__class__(
                page_content=chunk,
                metadata=doc.metadata
            ))
    return splitted_docs

In [7]:
def build_bm25_retriever(docs):
    """
    BM25Retriever oluşturur, 
    “Retriever”, arama kutusu gibi işlev görür: Onlarca, yüzlerce, belki binlerce sayfalık doküman içerisinden soruyla ilgili kısımları toplayıp modelin önüne getirir
    """
    retriever = BM25Retriever.from_documents(docs)
    return retriever

In [8]:
def create_qa_chain(retriever):
    """
    BM25Retriever ile çalışan bir RetrievalQA zinciri oluşturur.
    """
    #llm = OpenAI(temperature=0.0)  # veya model_name="gpt-3.5-turbo"
    llm=chatModel
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",           # "stuff" metodu: ilgili parçaları direkt birleştirir
        retriever=retriever,
        return_source_documents=True   # Kaynak dokümanları istiyorsanız True yapın
    )
    return qa_chain

In [9]:
folder_path = "PDF"  # PDF dosyalarının olduğu klasör
print("PDF’ler yükleniyor...")
documents = load_pdfs_from_folder(folder_path)
print(f"Toplam {len(documents)} doküman sayfası yüklendi.")

PDF’ler yükleniyor...
Toplam 83 doküman sayfası yüklendi.


In [10]:
print("Dokümanlar parçalara ayrılıyor...")
splitted_docs = split_documents(documents, chunk_size=1000, chunk_overlap=100)
print(f"Parçalanmış doküman sayısı: {len(splitted_docs)}")

print("BM25 tabanlı retriever oluşturuluyor...")
retriever = build_bm25_retriever(splitted_docs)

print("RetrievalQA zinciri oluşturuluyor...")
qa_chain = create_qa_chain(retriever)

Dokümanlar parçalara ayrılıyor...
Parçalanmış doküman sayısı: 245
BM25 tabanlı retriever oluşturuluyor...
RetrievalQA zinciri oluşturuluyor...


In [11]:
query = "Soru:aidatı ödemeyen kiracı ile ilgili ev sahibinin sorumluluğu nedir "


#result = qa_chain(query)
result = qa_chain.invoke({"query": query})
answer = result["result"]
source_docs = result["source_documents"]

print("\nCevap:")
print(answer)
print("\nKullanılan Kaynaklar:")
for i, doc in enumerate(source_docs, start=1):
    print(f"  {i}. {doc.metadata}")
print("-"*50)


Cevap:
Ev sahibinin sorumluluğu, kiracının aidat ödememesi durumunda Kat Mülkiyeti Kanunu'na göre düzenlenmiştir. Kiracı, bağımsız bölümde oturan bir kişi olarak kat malikleri gibi hak ve yükümlülüklere uymak zorundadır. Eğer kiracı aidatları ödemezse, ev sahibi, kiracının bu yükümlülüğünü yerine getirmesinden sorumlu tutulabilir. Ayrıca, aidat ödemeyen kiracı hakkında diğer kat malikleri veya yönetici tarafından yasal işlemler başlatılabilir. Ancak, ev sahibinin kiracıdan alacakları konusunda öncelik, aidat borcu olmayan kat maliklerine aittir. Bu nedenle, ev sahibi kiracının borçlarını tahsil etmekle yükümlü olabilir.

Kullanılan Kaynaklar:
  1. {'source': 'PDF\\apartmanyonetimplani.pdf', 'page': 5}
  2. {'source': 'PDF\\kat-mulkiyeti-kanununda-apartman-yoneticisinin-sorumlulugu-0f6cb6.pdf', 'page': 20}
  3. {'source': 'PDF\\kat-mulkiyeti-kanununda-apartman-yoneticisinin-sorumlulugu-0f6cb6.pdf', 'page': 10}
  4. {'source': 'PDF\\1.5.634.pdf', 'page': 8}
-----------------------------